In [27]:
import numpy as np
import pandas as pd

# Reproducibility
rng = np.random.default_rng(123)
N = 6000  # number of tenants

ages = rng.integers(21, 55, N)
city_tiers = rng.choice(["Tier1", "Tier2", "Tier3"], N, p=[0.45, 0.35, 0.20])
employer_types = rng.choice(["MNC", "SME", "Startup", "Gig"], N, p=[0.35, 0.25, 0.2, 0.2])
income_monthly = rng.normal(60000, 18000, N).clip(12000, 200000).astype(int)

rent_amount = (income_monthly * rng.uniform(0.15, 0.5, N)).astype(int)
deposit_multiple = rng.choice([0, 1, 2, 3], N, p=[0.1, 0.3, 0.45, 0.15])
lease_term = rng.integers(6, 24, N)
tenure_months_on_platform = rng.integers(0, 18, N)

avg_days_late = rng.poisson(2, N)
late_payment_rate = rng.beta(2, 6, N)
failed_txn_rate = rng.beta(1.2, 10, N)
payment_method = rng.choice(["UPI", "Card", "NetBanking", "Wallet"], N, p=[0.55, 0.25, 0.15, 0.05])

rent_to_income = rent_amount / income_monthly

# stronger PD signal
base_pd = (
    0.02
    + 0.35 * (rent_to_income > 0.45)
    + 0.12 * (city_tiers == "Tier3")
    + 0.10 * (employer_types == "Gig")
    + 0.07 * (late_payment_rate > 0.25)
    + 0.10 * (failed_txn_rate > 0.10)
    + 0.05 * (avg_days_late > 3)
    - 0.03 * (tenure_months_on_platform > 6)
    - 0.02 * (deposit_multiple >= 2)
)
base_pd = np.clip(base_pd + rng.normal(0, 0.01, N), 0.001, 0.9)
default_flag = rng.binomial(1, base_pd)

dpd_next_month = np.where(
    default_flag == 1,
    rng.integers(30, 120, N),
    rng.poisson(2, N)
)

df = pd.DataFrame({
    "tenant_id": np.arange(1, N+1),
    "age": ages,
    "city_tier": city_tiers,
    "employer_type": employer_types,
    "income_monthly": income_monthly,
    "rent_amount": rent_amount,
    "deposit_multiple": deposit_multiple,
    "lease_term_months": lease_term,
    "tenure_months_on_platform": tenure_months_on_platform,
    "avg_days_late": avg_days_late,
    "late_payment_rate": late_payment_rate,
    "failed_txn_rate": failed_txn_rate,
    "payment_method": payment_method,
    "rent_to_income": rent_to_income,
    "default_flag": default_flag,
    "dpd_next_month": dpd_next_month
})

df.to_csv("synthetic_rent_credit_dataset_v2.csv", index=False)
print("✅ Saved synthetic_rent_credit_dataset_v2.csv")


✅ Saved synthetic_rent_credit_dataset_v2.csv


In [28]:
df

,tenant_id,age,city_tier,employer_type,income_monthly,rent_amount,deposit_multiple,lease_term_months,tenure_months_on_platform,avg_days_late,late_payment_rate,failed_txn_rate,payment_method,rent_to_income,default_flag,dpd_next_month
0,1,21,Tier2,MNC,83967,19695,1,7,5,1,0.185669,0.135204,Card,0.234556,0,5
1,2,44,Tier2,Startup,85904,12995,2,15,13,5,0.167579,0.192587,UPI,0.151274,0,1
2,3,41,Tier2,Startup,68478,24198,3,6,2,3,0.353976,0.028753,Card,0.353369,0,2
3,4,22,Tier1,Startup,49144,16322,2,6,10,0,0.174880,0.082237,Card,0.332126,0,2
4,5,51,Tier2,MNC,60288,22185,0,6,15,0,0.360357,0.139243,UPI,0.367984,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5995,5996,36,Tier3,Startup,43426,14380,2,14,16,2,0.240690,0.030650,NetBanking,0.331138,0,3
5996,5997,28,Tier3,MNC,56554,12045,0,20,6,4,0.381313,0.041814,UPI,0.212982,1,36
5997,5998,22,Tier1,Startup,50626,17562,1,6,13,1,0.312652,0.046214,UPI,0.346897,0,2
5998,5999,29,Tier2,MNC,56876,14398,3,20,13,2,0.212283,0.079804,Card,0.253147,0,2


In [29]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["tenant_id", "default_flag", "dpd_next_month"])
y_cls = df["default_flag"]
y_reg = df["dpd_next_month"]

X_train, X_test, y_cls_train, y_cls_test = train_test_split(
    X, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)

In [30]:
X_train

,age,city_tier,employer_type,income_monthly,rent_amount,deposit_multiple,lease_term_months,tenure_months_on_platform,avg_days_late,late_payment_rate,failed_txn_rate,payment_method,rent_to_income
1400,25,Tier1,MNC,59117,29037,1,10,9,3,0.290322,0.116400,UPI,0.491179
1564,50,Tier1,Gig,50605,16889,1,17,5,4,0.264710,0.035328,UPI,0.333742
1640,21,Tier1,Gig,86798,21946,2,21,8,1,0.201874,0.050591,UPI,0.252840
2768,36,Tier1,MNC,54392,10106,3,7,7,2,0.315846,0.139863,UPI,0.185799
447,54,Tier3,Startup,63152,31236,1,12,5,0,0.170003,0.028443,UPI,0.494616
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1176,24,Tier2,Startup,52713,20909,1,6,15,3,0.347295,0.056195,UPI,0.396657
4094,49,Tier3,Startup,52813,19005,2,23,4,1,0.338097,0.007849,UPI,0.359855
5812,29,Tier2,Gig,97484,40379,1,19,15,3,0.191534,0.012903,UPI,0.414212
1725,42,Tier1,Gig,63327,14374,2,21,17,1,0.119530,0.166560,NetBanking,0.226981


In [31]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, mean_absolute_error, mean_squared_error
import lightgbm as lgb

# 1) Load data
df = pd.read_csv("/Users/anshagarwal/Desktop/KirayaEase/src/synthetic_rent_credit_dataset_v2.csv")

# 2) Define features/targets
cat = ["city_tier", "employer_type", "payment_method"]
num = [
    "age","income_monthly","rent_amount","deposit_multiple","lease_term_months",
    "tenure_months_on_platform","avg_days_late","late_payment_rate",
    "failed_txn_rate","rent_to_income"
]
y_cls = df["default_flag"].astype(int)
y_reg = df["dpd_next_month"]

X = df[cat + num]

# 3) Train/test split (tenant-level; one row per tenant)
X_train, X_test, y_cls_train, y_cls_test = train_test_split(
    X, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)
_, _, y_reg_train, y_reg_test = train_test_split(
    X, y_reg, test_size=0.2, random_state=42  # same split seed to align with above
)

# 4) Preprocessor: OneHot for categoricals, passthrough numerics
pre = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat),
        ("num", "passthrough", num),
    ],
    remainder="drop",
)

# 5) Classifier: LightGBM + calibration
clf = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1,
)

pipe_clf = Pipeline([("pre", pre), ("lgbm", clf)])
cal = CalibratedClassifierCV(pipe_clf, method="isotonic", cv=3)
cal.fit(X_train, y_cls_train)

p_test = cal.predict_proba(X_test)[:, 1]
print("AUROC:", roc_auc_score(y_cls_test, p_test))
print("PR-AUC:", average_precision_score(y_cls_test, p_test))
print("Brier:", brier_score_loss(y_cls_test, p_test))

# 6) Regressor: LightGBM for dpd_next_month
reg = lgb.LGBMRegressor(
    n_estimators=600,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)
pipe_reg = Pipeline([("pre", pre), ("lgbm", reg)])
pipe_reg.fit(X_train, y_reg_train)

preds = pipe_reg.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_reg_test, preds))
mae = mean_absolute_error(y_reg_test, preds)
print("RMSE:", rmse)
print("MAE:", mae)


[LightGBM] [Info] Number of positive: 540, number of negative: 2660
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000595 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1382
[LightGBM] [Info] Number of data points in the train set: 3200, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


/Users/anshagarwal/Desktop/KirayaEase/.venv311/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 540, number of negative: 2660
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000221 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1382
[LightGBM] [Info] Number of data points in the train set: 3200, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


/Users/anshagarwal/Desktop/KirayaEase/.venv311/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 540, number of negative: 2660
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000301 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1382
[LightGBM] [Info] Number of data points in the train set: 3200, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


/Users/anshagarwal/Desktop/KirayaEase/.venv311/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/anshagarwal/Desktop/KirayaEase/.venv311/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/anshagarwal/Desktop/KirayaEase/.venv311/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/anshagarwal/Desktop/KirayaEase/.venv311/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


AUROC: 0.6881138514653069
PR-AUC: 0.31137634339889353
Brier: 0.13124743462458374
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000282 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1382
[LightGBM] [Info] Number of data points in the train set: 4800, number of used features: 21
[LightGBM] [Info] Start training from score 13.975208
RMSE: 31.65791525820414
MAE: 21.53807610019192


/Users/anshagarwal/Desktop/KirayaEase/.venv311/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [32]:
results_cls = X_test.copy()
results_cls["tenant_id"] = df.loc[X_test.index, "tenant_id"]
results_cls["actual_default"] = y_cls_test.values
results_cls["pd_pred"] = p_test  # predicted probability of default

In [33]:
results_cls

,city_tier,employer_type,payment_method,age,income_monthly,rent_amount,deposit_multiple,lease_term_months,tenure_months_on_platform,avg_days_late,late_payment_rate,failed_txn_rate,rent_to_income,tenant_id,actual_default,pd_pred
5783,Tier2,SME,UPI,41,55300,9596,0,21,15,1,0.284914,0.033021,0.173526,5784,0,0.087113
675,Tier2,MNC,NetBanking,52,67534,29789,0,9,6,3,0.183308,0.275857,0.441096,676,0,0.181093
5802,Tier1,MNC,NetBanking,53,43216,16963,1,21,0,3,0.375282,0.261864,0.392517,5803,0,0.225951
1895,Tier2,SME,UPI,33,81685,39685,2,14,10,4,0.102036,0.244359,0.485830,1896,1,0.393690
4689,Tier1,Gig,UPI,26,55189,17618,1,21,2,1,0.318430,0.032997,0.319230,4690,0,0.075039
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5777,Tier3,SME,UPI,53,38694,14175,2,15,3,0,0.512666,0.019073,0.366336,5778,0,0.188989
2012,Tier3,MNC,NetBanking,35,48146,13326,0,6,11,4,0.128622,0.008723,0.276783,2013,0,0.072012
5405,Tier1,MNC,Card,43,95392,21790,2,12,13,3,0.276831,0.063688,0.228426,5406,0,0.059938
1450,Tier2,MNC,UPI,26,33383,11672,1,14,16,1,0.127807,0.034930,0.349639,1451,0,0.045076


In [ ]:
# Apply business rules (cutoffs)
def decide(pd_hat: float) -> str:
    if pd_hat < 0.03: 
        return "APPROVE_FULL"
    if pd_hat < 0.08: 
        return "APPROVE_PARTIAL"
    return "DECLINE"

results_cls["decision"] = results_cls["pd_pred"].apply(decide)


In [35]:
results_cls

,city_tier,employer_type,payment_method,age,income_monthly,rent_amount,deposit_multiple,lease_term_months,tenure_months_on_platform,avg_days_late,late_payment_rate,failed_txn_rate,rent_to_income,tenant_id,actual_default,pd_pred,decision
5783,Tier2,SME,UPI,41,55300,9596,0,21,15,1,0.284914,0.033021,0.173526,5784,0,0.087113,DECLINE
675,Tier2,MNC,NetBanking,52,67534,29789,0,9,6,3,0.183308,0.275857,0.441096,676,0,0.181093,DECLINE
5802,Tier1,MNC,NetBanking,53,43216,16963,1,21,0,3,0.375282,0.261864,0.392517,5803,0,0.225951,DECLINE
1895,Tier2,SME,UPI,33,81685,39685,2,14,10,4,0.102036,0.244359,0.485830,1896,1,0.393690,DECLINE
4689,Tier1,Gig,UPI,26,55189,17618,1,21,2,1,0.318430,0.032997,0.319230,4690,0,0.075039,APPROVE_PARTIAL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5777,Tier3,SME,UPI,53,38694,14175,2,15,3,0,0.512666,0.019073,0.366336,5778,0,0.188989,DECLINE
2012,Tier3,MNC,NetBanking,35,48146,13326,0,6,11,4,0.128622,0.008723,0.276783,2013,0,0.072012,APPROVE_PARTIAL
5405,Tier1,MNC,Card,43,95392,21790,2,12,13,3,0.276831,0.063688,0.228426,5406,0,0.059938,APPROVE_PARTIAL
1450,Tier2,MNC,UPI,26,33383,11672,1,14,16,1,0.127807,0.034930,0.349639,1451,0,0.045076,APPROVE_PARTIAL
